# SOA ML281 - AI and Individual Life Insurance Challenge

## Reproduction notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Society-of-actuaries-research-institute/ML281-AI-and-Individual-Life-Insurance-Challenge-Pramod-Misra/blob/main/notebooks/ML281_reproduce.ipynb)

**Author:** Pramod Misra, Independent Researcher, Georgia Institute of Technology, Atlanta
**Contact:** pmisra30@gatech.edu
**Licence:** GPL-3.0 (see `LICENSE`)

This notebook regenerates every figure and every published number in the ML281
submission. **Runtime -> Run all.** No uploads, no Google Drive, no BigQuery, no
credentials. The input sample downloads once (about 6 MB) from a public Hugging
Face dataset and is checksum-verified before use.

Each expensive result (model fits, cross-validation, the copula simulation) is
computed **once** and reused by the figures. A full run measured about 8 minutes
end to end on a laptop CPU; the multi-model cross-validation is the slow cell. A
GPU does not help.

Outputs land in two places:

| Directory | Contents |
|---|---|
| `figures/` | the seven published figures, as PNG |
| `results/` | one JSON per section -- the numeric ledger behind `RESULTS.md` |

All seeds are 42. If you ever see `ImportError: cannot import name '_center'`,
that is a stale-numpy issue in a live kernel -- do **Runtime -> Restart session**
and Run all again.

> **Smoke mode.** Setting the environment variable `ML281_SMOKE=1` runs a fast,
> shrunken version against the small committed fixture in `data/sample/`. That is
> what CI executes to prove the notebook still runs. Its output is written to
> `.smoke/` so it can never overwrite the published figures and results. Smoke
> output is **not** a reproduction of the published numbers -- leave the variable
> unset for that.

## 1. Install

On Colab this pins the modelling libraries (`xgboost`, `scikit-learn`, `shap`,
`pyarrow`) but deliberately leaves the preinstalled `numpy` / `scipy` / `pandas`
alone: re-pinning numpy inside a live kernel is what triggers the `_center`
`ImportError`. The exact resolved versions are recorded in
`results/environment.json`, and the next cell warns if they differ from the
reference environment that produced the published figures.

Locally, install the full pinned set instead -- `pip install -r requirements.txt`
-- which reproduces the reference environment exactly.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules

# Versions that produced the published figures and results.
REFERENCE_VERSIONS = {
    "numpy": "2.4.2",
    "pandas": "2.3.3",
    "scipy": "1.17.0",
    "scikit-learn": "1.8.0",
    "xgboost": "3.2.0",
    "pyarrow": "23.0.1",
    "shap": "0.51.0",
}

if IN_COLAB:
    # Safe to pin mid-kernel: none of these are imported yet, and none of them
    # are ABI-coupled to the running numpy the way scipy/pandas are.
    _pins = ["xgboost==3.2.0", "scikit-learn==1.8.0", "shap==0.51.0", "pyarrow==23.0.1"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pins], check=False)
    print("Colab: pinned modelling stack; numpy/scipy/pandas left as preinstalled.")
else:
    print("Local: expecting `pip install -r requirements.txt` to have been run.")

## 2. Setup and data load

Fetches the public sample into `data/` (once), verifies its SHA-256, loads it,
builds the shared 60/20/20 split, and trains the one anchored XGBoost model that
the ablation, the offset comparison and Figures 2 and 5 all reuse.

In [ ]:
import hashlib, json, os, platform, urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, mean_poisson_deviance)
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

SEED = 42
np.random.seed(SEED)

SMOKE = os.environ.get("ML281_SMOKE", "").strip().lower() in {"1", "true", "yes"}

# Every expensive knob in one place, so smoke mode is one dict update.
K = dict(n_est=300, reg_rounds=2000, folds=5, sims=2500, rf_est=200,
         mlp_iter=500, shap_n=2000, oot_rounds=300)
if SMOKE:
    K.update(n_est=30, reg_rounds=100, folds=2, sims=200, rf_est=20,
             mlp_iter=30, shap_n=300, oot_rounds=30)
    print("*** SMOKE MODE: shrunken run on the committed fixture. "
          "Numbers are NOT the published results. ***")

# --- locate the repo root so the notebook works from notebooks/ or from Colab ---
_here = os.path.abspath(os.getcwd())
ROOT = _here
for _ in range(4):                                        # MAX 4 levels up
    if os.path.isdir(os.path.join(ROOT, "data")) or os.path.isdir(os.path.join(ROOT, ".git")):
        break
    _parent = os.path.dirname(ROOT)
    if _parent == ROOT:
        break
    ROOT = _parent
if os.path.basename(_here) == "notebooks":
    ROOT = os.path.dirname(_here)

DATA_DIR = os.path.join(ROOT, "data"); os.makedirs(DATA_DIR, exist_ok=True)
# Smoke output goes to .smoke/ so a fast CI run can never overwrite the published
# figures and results committed at the repo root.
_OUT_ROOT = os.path.join(ROOT, ".smoke") if SMOKE else ROOT
FIG_DIR = os.path.join(_OUT_ROOT, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
RESULTS_DIR = os.path.join(_OUT_ROOT, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)
print("figures ->", FIG_DIR)
print("results ->", RESULTS_DIR)

# --- fetch + verify the inputs (public Hugging Face dataset, no credentials) ---
_BASE = "https://huggingface.co/datasets/pramodmisra/soa-ml281-sample/resolve/main/"
SHA256 = {
    "soa_sample.parquet": "9bcd09b4ef7b25e8c5a9dfe8a6723502b1cd5541af2b8c5b49a5d004793c2e9a",
    "full_table_ae.json": "3a2e9b2e14758210d50a00cff23eceb3b909ed255852311f0cc443e1db41100f",
}

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def ensure(fn):
    """Download `fn` into data/ if absent, then verify its checksum."""
    dst = os.path.join(DATA_DIR, fn)
    if not os.path.exists(dst):
        print("downloading", fn, "...")
        urllib.request.urlretrieve(_BASE + fn, dst)
    got = sha256_file(dst)
    if got != SHA256[fn]:
        raise RuntimeError(f"{fn} sha256 mismatch\n  expected {SHA256[fn]}\n  got      {got}")
    return dst

FULL_AE = ensure("full_table_ae.json")
if SMOKE:
    SAMPLE = os.path.join(DATA_DIR, "sample", "soa_sample_small.parquet")
    if not os.path.exists(SAMPLE):
        raise FileNotFoundError(f"smoke fixture missing: {SAMPLE}")
else:
    SAMPLE = ensure("soa_sample.parquet")

# dpi 170 matches the raster of the published figures. Keep it in step with
# the committed PNGs in figures/ -- changing it changes every figure's size.
plt.rcParams.update({"figure.dpi": 170, "savefig.dpi": 170, "font.size": 9,
                     "axes.titlesize": 10, "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE = "#2c5282", "#dd6b20"

# --- results ledger -----------------------------------------------------------
def _jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o)}")

def emit(name, payload):
    """Write one section's numbers to results/<name>.json."""
    path = os.path.join(RESULTS_DIR, name + ".json")
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, sort_keys=True, default=_jsonable)
    print("wrote " + os.path.relpath(path, ROOT).replace(os.sep, "/"))

df = pd.read_parquet(SAMPLE)
for c in ["Sex", "Smoker_Status", "Insurance_Plan"]:
    df[c] = LabelEncoder().fit_transform(df[c].astype(str))
FEATS = ["Issue_Age", "Attained_Age", "Duration", "Sex", "Smoker_Status", "Insurance_Plan",
         "Log_Amount_Exposed", "Log_Policies_Exposed", "Expected_Mortality_Rate"]
BASE = [c for c in FEATS if c != "Expected_Mortality_Rate"]
ANCHORED = FEATS
y = df["Death_Count"].astype(float)
idx = np.arange(len(df))
i_tr, i_tmp = train_test_split(idx, test_size=0.4, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=0.5, random_state=SEED)

# Splits must be disjoint -- guard against a silent leak.
assert len(set(i_tr) & set(i_te)) == 0, "train/test overlap"
assert len(set(i_tr) & set(i_va)) == 0, "train/val overlap"
assert len(set(i_va) & set(i_te)) == 0, "val/test overlap"

exp_vbt = df["ExpDth_VBT2015wMI_Cnt"].astype(float).to_numpy()
pol = df["Policies_Exposed"].astype(float).to_numpy()

SUBMIT = dict(n_estimators=K["n_est"], learning_rate=0.05, max_depth=6, subsample=0.8,
              colsample_bytree=0.8, random_state=SEED, objective="reg:squarederror")
REG = dict(n_estimators=K["reg_rounds"], learning_rate=0.05, max_depth=6, subsample=0.8,
           colsample_bytree=0.8, min_child_weight=5, reg_lambda=2.0, reg_alpha=0.5, gamma=0.1,
           random_state=SEED, objective="reg:squarederror", early_stopping_rounds=50)

# Train the anchored model ONCE; reused by Q1, Q4 baseline, and Figures 2 & 5.
model_anch = xgb.XGBRegressor(**SUBMIT).fit(df[ANCHORED].iloc[i_tr], y.iloc[i_tr])
print(f"Loaded {len(df):,} cells | split {len(i_tr):,}/{len(i_va):,}/{len(i_te):,} | anchored model trained")

# --- environment record + drift warning --------------------------------------
import importlib.metadata as _md
_resolved = {}
for _p in REFERENCE_VERSIONS:
    try:
        _resolved[_p] = _md.version(_p)
    except Exception:
        _resolved[_p] = "not installed"
_drift = {p: (REFERENCE_VERSIONS[p], _resolved[p]) for p in REFERENCE_VERSIONS
          if _resolved[p] != REFERENCE_VERSIONS[p]}
if _drift:
    print("\nWARNING: package versions differ from the reference environment.")
    for _p, (_ref, _got) in _drift.items():
        print(f"  {_p}: reference {_ref}, running {_got}")
    print("  Tree-ensemble and SHAP output are not guaranteed bit-identical across versions.")
else:
    print("\nEnvironment matches the reference versions exactly.")

emit("environment", {
    "seed": SEED, "smoke_mode": SMOKE, "knobs": K,
    "python": platform.python_version(), "platform": platform.platform(),
    "reference_versions": REFERENCE_VERSIONS, "resolved_versions": _resolved,
    "version_drift": {p: {"reference": r, "running": g} for p, (r, g) in _drift.items()},
    "input_file": os.path.basename(SAMPLE),
    "input_sha256": sha256_file(SAMPLE),
    "n_cells": int(len(df)),
    "split_sizes": {"train": int(len(i_tr)), "val": int(len(i_va)), "test": int(len(i_te))},
})

### Full-table A/E (the 45.5M-cell population)

`data/full_table_ae.json` holds pre-aggregated actual/expected counts for the
**entire** ILEC 2012-2019 table, not the sample. It is the source for Figure 1 and
for the headline population A/E quoted in the report. Recomputing the totals here
is a check on that number.

In [ ]:
_ae = json.load(open(FULL_AE, encoding="utf-8"))
_by_sex = pd.DataFrame(_ae["Sex"])
_deaths = float(_by_sex["deaths"].sum())
_expected = float(_by_sex["exp"].sum())
FULL_AE_SUMMARY = {
    "total_deaths": _deaths,
    "total_expected_vbt2015wmi": _expected,
    "overall_ae": round(_deaths / _expected, 4),
    "segments": {k: len(v) for k, v in _ae.items()},
}
print(f"full table: deaths {_deaths:,.0f} | expected {_expected:,.0f} | overall A/E "
      f"{FULL_AE_SUMMARY['overall_ae']}")
emit("full_table_ae_summary", FULL_AE_SUMMARY)

## 3. Q1 - VBT-feature ablation

Reuses the anchored model and trains only the un-anchored one. Removing the VBT
feature does not reduce performance (test R2 about 0.775 without vs 0.759 with),
which is the point: the model is not merely relearning the table. (REPORT 4.3)

In [ ]:
def fit_eval(cols, params, tag, early=False):
    Xtr, ytr = df[cols].iloc[i_tr], y.iloc[i_tr]
    Xva, yva = df[cols].iloc[i_va], y.iloc[i_va]
    Xte, yte = df[cols].iloc[i_te], y.iloc[i_te]
    m = xgb.XGBRegressor(**params)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False) if early else m.fit(Xtr, ytr)
    out = {"model": tag}
    for nm, Xs, ys in [("train", Xtr, ytr), ("val", Xva, yva), ("test", Xte, yte)]:
        p = np.clip(m.predict(Xs), 0, None)
        out[f"{nm}_RMSE"] = round(np.sqrt(mean_squared_error(ys, p)), 4)
        out[f"{nm}_R2"] = round(r2_score(ys, p), 4)
    p_te = np.clip(m.predict(Xte), 0, None)
    out["test_MAE"] = round(mean_absolute_error(yte, p_te), 4)
    out["AE_model"] = round(yte.sum() / p_te.sum(), 4)
    out["AE_VBT"] = round(yte.sum() / exp_vbt[i_te].sum(), 4)
    if early:
        out["best_iter"] = int(m.best_iteration)
    return out, m

def _scores(model, cols):
    out = {"model": "anchored (with VBT)"}
    for nm, ii in [("train", i_tr), ("val", i_va), ("test", i_te)]:
        p = np.clip(model.predict(df[cols].iloc[ii]), 0, None)
        out[f"{nm}_RMSE"] = round(np.sqrt(mean_squared_error(y.iloc[ii], p)), 4)
        out[f"{nm}_R2"] = round(r2_score(y.iloc[ii], p), 4)
    p_te = np.clip(model.predict(df[cols].iloc[i_te]), 0, None)
    out["AE_model"] = round(y.iloc[i_te].sum() / p_te.sum(), 4)
    out["AE_VBT"] = round(y.iloc[i_te].sum() / exp_vbt[i_te].sum(), 4)
    return out

r_anch = _scores(model_anch, ANCHORED)              # reuse the model trained in setup
r_noanch, _ = fit_eval(BASE, SUBMIT, "un-anchored (no VBT)")
ABL = (r_anch["test_R2"], r_noanch["test_R2"])       # (with VBT, without VBT) -> Figure 4 left
print(pd.DataFrame([r_anch, r_noanch]).to_string(index=False))

emit("ablation", {"with_vbt": r_anch, "without_vbt": r_noanch,
                  "test_r2_with_vbt": ABL[0], "test_r2_without_vbt": ABL[1]})

## 4. Q2 - generalization (regularized fit, 5-fold CV, error by band)

Trains the regularized model once and reuses it for the band table. (REPORT 4.1-4.2)

In [ ]:
r_reg, m_reg = fit_eval(ANCHORED, REG, "regularized+earlystop", early=True)
print(pd.DataFrame([r_reg]).to_string(index=False))

kf = KFold(n_splits=K["folds"], shuffle=True, random_state=SEED)
cv_rmse, cv_mae, cv_r2 = [], [], []
for tr, te in kf.split(df):
    m = xgb.XGBRegressor(**SUBMIT)
    m.fit(df[ANCHORED].iloc[tr], y.iloc[tr])
    p = np.clip(m.predict(df[ANCHORED].iloc[te]), 0, None)
    cv_rmse.append(np.sqrt(mean_squared_error(y.iloc[te], p)))
    cv_mae.append(mean_absolute_error(y.iloc[te], p)); cv_r2.append(r2_score(y.iloc[te], p))
print(f"\n{K['folds']}-fold CV: RMSE {np.mean(cv_rmse):.4f} +/- {np.std(cv_rmse):.4f} | "
      f"MAE {np.mean(cv_mae):.4f} +/- {np.std(cv_mae):.4f} | R2 {np.mean(cv_r2):.4f} +/- {np.std(cv_r2):.4f}")

p_te = np.clip(m_reg.predict(df[ANCHORED].iloc[i_te]), 0, None)   # reuse m_reg (no refit)
yte = y.iloc[i_te].to_numpy()
band = pd.cut(yte, [-1, 0, 1, 3, 10, np.inf], labels=["0", "1", "2-3", "4-10", "11+"])
bd = pd.DataFrame({"band": band, "sq": (yte - p_te) ** 2, "ae": np.abs(yte - p_te)})
band_tbl = bd.groupby("band", observed=True).agg(
    n=("sq", "size"), rmse=("sq", lambda s: round(float(np.sqrt(s.mean())), 4)),
    mae=("ae", lambda s: round(float(s.mean()), 4)))
print("\nError by Death_Count band:")
print(band_tbl.to_string())

emit("generalization", {
    "regularized": r_reg,
    "cv_folds": K["folds"],
    "cv_r2_mean": round(float(np.mean(cv_r2)), 4), "cv_r2_sd": round(float(np.std(cv_r2)), 4),
    "cv_rmse_mean": round(float(np.mean(cv_rmse)), 4), "cv_rmse_sd": round(float(np.std(cv_rmse)), 4),
    "cv_mae_mean": round(float(np.mean(cv_mae)), 4), "cv_mae_sd": round(float(np.std(cv_mae)), 4),
    "error_by_death_count_band": band_tbl.reset_index().to_dict(orient="records"),
})

## 5. Multi-model comparison (5-fold cross-validation) - reused by Figure 3

GLM / NN / RF / XGBoost under a single shared 5-fold split. This is the heaviest
cell; it runs once and Figure 3 replots the stored numbers. The headline finding is
that the differences sit inside fold-to-fold noise. (REPORT 3.1)

In [ ]:
Xv = df[FEATS].to_numpy(float); yv = df["Death_Count"].to_numpy(float)
kf5 = KFold(n_splits=K["folds"], shuffle=True, random_state=SEED)
_specs = [("GLM (Poisson)", "GLM\n(Poisson)", "scaled", lambda: PoissonRegressor(alpha=1e-4, max_iter=2000)),
          ("Neural Net (64-32-16)", "Neural\nNet", "scaled", lambda: MLPRegressor(hidden_layer_sizes=(64,32,16), random_state=SEED, max_iter=K["mlp_iter"], early_stopping=True)),
          ("Random Forest", "Random\nForest", "raw", lambda: RandomForestRegressor(n_estimators=K["rf_est"], max_depth=12, n_jobs=-1, random_state=SEED)),
          ("XGBoost", "XGBoost", "raw", lambda: xgb.XGBRegressor(n_estimators=K["n_est"], learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=SEED, objective="reg:squarederror"))]
MM_NUM = []; _rows = []
for full, label, mode, ctor in _specs:
    a, b = [], []
    for tri, tei in kf5.split(Xv):
        Xtr, Xte = Xv[tri], Xv[tei]; ytr, yte_ = yv[tri], yv[tei]
        if mode == "scaled":
            sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
        m = ctor(); m.fit(Xtr, ytr); p = np.clip(m.predict(Xte), 0, None)
        a.append(r2_score(yte_, p)); b.append(np.sqrt(mean_squared_error(yte_, p)))
    MM_NUM.append(dict(label=label, name=full, r2m=float(np.mean(a)), r2s=float(np.std(a)),
                       rmm=float(np.mean(b)), rms=float(np.std(b))))
    _rows.append({"model": full, "cv_R2": f"{np.mean(a):.4f} +/- {np.std(a):.4f}",
                  "cv_RMSE": f"{np.mean(b):.4f} +/- {np.std(b):.4f}"})
print(pd.DataFrame(_rows).to_string(index=False))

emit("multimodel", {"cv_folds": K["folds"], "models": [
    {"model": m["name"], "cv_r2_mean": round(m["r2m"], 4), "cv_r2_sd": round(m["r2s"], 4),
     "cv_rmse_mean": round(m["rmm"], 4), "cv_rmse_sd": round(m["rms"], 4)} for m in MM_NUM]})

## 6. Q4 - Poisson exposure / VBT offset - reused by Figure 4 (right)

The submitted squared-error model is compared against two properly specified count
models, each carrying a log offset (exposure, then the VBT expected count). The
baseline deviance reuses the anchored model. (REPORT 4.4)

In [ ]:
def poisson_offset(offset_col, tag):
    Xtr, Xva, Xte = df[BASE].iloc[i_tr], df[BASE].iloc[i_va], df[BASE].iloc[i_te]
    off = np.log(np.clip(offset_col, 1e-9, None))
    dtr = xgb.DMatrix(Xtr, label=y.iloc[i_tr]); dtr.set_base_margin(off[i_tr])
    dva = xgb.DMatrix(Xva, label=y.iloc[i_va]); dva.set_base_margin(off[i_va])
    dte = xgb.DMatrix(Xte, label=y.iloc[i_te]); dte.set_base_margin(off[i_te])
    params = dict(objective="count:poisson", eta=0.05, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, eval_metric="poisson-nloglik", seed=SEED)
    bst = xgb.train(params, dtr, num_boost_round=K["reg_rounds"], evals=[(dva, "val")],
                    early_stopping_rounds=50, verbose_eval=False)
    p = np.clip(bst.predict(dte), 1e-9, None); yte_ = y.iloc[i_te].to_numpy()
    return {"model": tag, "best_iter": bst.best_iteration,
            "test_RMSE": round(np.sqrt(mean_squared_error(yte_, p)), 4),
            "test_poisson_dev": round(mean_poisson_deviance(yte_, p), 4),
            "AE_model": round(yte_.sum() / p.sum(), 4)}

p_sub = np.clip(model_anch.predict(df[ANCHORED].iloc[i_te]), 1e-9, None)   # reuse anchored model
sub_dev = round(mean_poisson_deviance(y.iloc[i_te].to_numpy(), p_sub), 4)
o_exp = poisson_offset(pol, "Poisson + exposure offset")
o_vbt = poisson_offset(exp_vbt, "Poisson + VBT offset (A/E model)")
DEV = (sub_dev, o_exp["test_poisson_dev"], o_vbt["test_poisson_dev"])     # -> Figure 4 right
print(f"submitted squared-error model: test Poisson deviance = {sub_dev}")
print(pd.DataFrame([o_exp, o_vbt]).to_string(index=False))

emit("poisson_offset", {"submitted_squared_error_deviance": sub_dev,
                        "exposure_offset": o_exp, "vbt_offset": o_vbt})

## 7. Q6 - frequency-severity copula (aggregate loss / TVaR) - reused by Figure 6

A Gaussian copula couples cell-level claim frequency and average severity, then the
aggregate loss distribution is simulated to read off VaR99 and TVaR99. The
per-iteration loop has a fixed bound. (REPORT 4.7)

In [ ]:
from scipy.stats import norm, rankdata
rng = np.random.default_rng(SEED)
d = df[(df["Death_Count"] > 0) & (df["Death_Claim_Amount"] > 0)]
freq = d["Death_Count"].to_numpy(float); sev = (d["Death_Claim_Amount"] / d["Death_Count"]).to_numpy(float)
u = np.column_stack([(rankdata(freq) - 0.5) / len(freq), (rankdata(sev) - 0.5) / len(sev)])
z = norm.ppf(np.clip(u, 1e-6, 1 - 1e-6)); rho = float(np.corrcoef(z, rowvar=False)[0, 1])
N, SIMS = len(d), K["sims"]
L = np.linalg.cholesky([[1, rho], [rho, 1]]); agg = np.empty(SIMS)
fq_sorted = np.sort(freq); sv_sorted = np.sort(sev)            # sort once; quantile via interpolation
qp = (np.arange(N) + 0.5) / N
for s in range(SIMS):                                          # MAX K["sims"] iterations
    g = rng.standard_normal((N, 2)) @ L.T; uu = norm.cdf(g)
    agg[s] = float(np.sum(np.interp(uu[:, 0], qp, fq_sorted) * np.interp(uu[:, 1], qp, sv_sorted)))
var99 = float(np.percentile(agg, 99)); tvar99 = float(agg[agg >= var99].mean())
COP = dict(rho=rho, agg=agg, mean=float(agg.mean()), var99=var99, tvar99=tvar99)
print(f"cells with deaths: {N:,} | freq-severity copula rho = {rho:.3f}")
print(f"aggregate claims: mean={agg.mean():,.0f}  VaR99={var99:,.0f}  TVaR99={tvar99:,.0f}")
print(f"TVaR99 / mean load = {tvar99/agg.mean():.3f}x")

emit("copula", {"rho": round(rho, 4), "n_cells_with_deaths": int(N), "sims": int(SIMS),
                "aggregate_mean": round(COP["mean"], 2), "var99": round(var99, 2),
                "tvar99": round(tvar99, 2),
                "tvar99_over_mean": round(tvar99 / COP["mean"], 4)})

## 8. Out-of-time validation (train <= 2017, test 2018-19) - reused by Figure 7

The honest test of a pricing model: fit on the past, score the future. The static
VBT2015 table lands closest to 1.0, which is the finding the report leads with.
(REPORT 4.8)

In [ ]:
tr = df[df.Observation_Year <= 2017]; te = df[df.Observation_Year >= 2018]
ytr, yte_o = tr.Death_Count.astype(float), te.Death_Count.astype(float)
exp_te = te.ExpDth_VBT2015wMI_Cnt.astype(float)
_XGB = dict(n_estimators=K["n_est"], learning_rate=0.05, max_depth=6, subsample=0.8,
            colsample_bytree=0.8, random_state=SEED)
def _ae(cols):
    m = xgb.XGBRegressor(**_XGB).fit(tr[cols], ytr)
    return round(yte_o.sum() / np.clip(m.predict(te[cols]), 0, None).sum(), 4)
ae_anch, ae_no = _ae(FEATS), _ae(BASE)
dtr = xgb.DMatrix(tr[BASE], label=ytr); dtr.set_base_margin(np.log(np.clip(tr.ExpDth_VBT2015wMI_Cnt.astype(float), 1e-9, None)).values)
dte = xgb.DMatrix(te[BASE], label=yte_o); dte.set_base_margin(np.log(np.clip(exp_te, 1e-9, None)).values)
b = xgb.train(dict(objective="count:poisson", eta=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric="poisson-nloglik", seed=SEED), dtr, K["oot_rounds"])
ae_off = round(yte_o.sum() / np.clip(b.predict(dte), 1e-9, None).sum(), 4)
ae_vbt = round(yte_o.sum() / exp_te.sum(), 4)
OOT_AE = dict(vbt=ae_vbt, anch=ae_anch, no=ae_no, off=ae_off)
res = pd.DataFrame([{"model": "VBT2015 (static)", "OOT_A/E": ae_vbt},
                    {"model": "Poisson VBT-offset", "OOT_A/E": ae_off},
                    {"model": "XGBoost (anchored)", "OOT_A/E": ae_anch},
                    {"model": "XGBoost (no VBT)", "OOT_A/E": ae_no}])
res["|A/E-1|"] = (res["OOT_A/E"] - 1).abs().round(4)
print(f"train {len(tr):,} cells / test {len(te):,} cells"); print(res.to_string(index=False))

emit("out_of_time", {"train_cells": int(len(tr)), "test_cells": int(len(te)),
                     "ae_vbt_static": ae_vbt, "ae_poisson_vbt_offset": ae_off,
                     "ae_xgboost_anchored": ae_anch, "ae_xgboost_no_vbt": ae_no,
                     "table": res.to_dict(orient="records")})

## 9. Figures - plot the already-computed results (fast; no retraining)

Only Figures 2 and 5 touch a model (the anchored one already in memory) plus the
SHAP values; the rest replot stored numbers.

In [ ]:
def _save(fig, name): fig.savefig(os.path.join(FIG_DIR, name)); plt.close(fig); print(name)

# Figure 1 - EDA from full-table aggregates
data = json.load(open(FULL_AE, encoding="utf-8"))
def _d(key, order=None):
    t = pd.DataFrame(data[key]).rename(columns={"deaths": "d", "exp": "e"})
    return t.set_index("k").reindex(order).reset_index() if order else t
sm, sx, age = _d("Smoker_Status"), _d("Sex"), _d("age_band", ['<40','40-49','50-59','60-69','70-79','80+'])
fig, ax = plt.subplots(2, 2, figsize=(10, 7))
def _bars(a, x, h, ttl, ylab, fmt="{:.3f}", line=True):
    bb = a.bar(x, h, color=BLUE)
    if line: a.axhline(1.0, color="red", ls="--", lw=1, label="A/E = 1.0"); a.legend(fontsize=7)
    a.set_title(ttl); a.set_ylabel(ylab)
    for r, v in zip(bb, h): a.text(r.get_x()+r.get_width()/2, v, fmt.format(v), ha="center", va="bottom", fontsize=8)
_bars(ax[0,0], [{"S":"Smoker","NS":"Non-Smoker","U":"Unknown"}.get(k,k) for k in sm.k], sm.ae.values, "A/E by Smoker Status (full data)", "A/E")
_bars(ax[0,1], sx.k.values, sx.ae.values, "A/E by Sex (full data)", "A/E")
_bars(ax[1,0], age.k.values, age.ae.values, "A/E by Attained-Age Band (full data)", "A/E")
_bars(ax[1,1], age.k.values, age.d.values/1e3, "Deaths by Attained-Age Band", "Deaths (000s)", fmt="{:.0f}", line=False)
fig.suptitle("Figure 1 - Mortality experience vs VBT2015 (full 45.5M-cell table)", fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.97]); _save(fig, "fig1_eda_ae_segments.png")

# Figure 2 - performance & importance (reuse model_anch)
Xte = df[ANCHORED].iloc[i_te]; yte = y.iloc[i_te]
rmse = {k: np.sqrt(mean_squared_error(y.iloc[ii], np.clip(model_anch.predict(df[ANCHORED].iloc[ii]),0,None))) for k, ii in [("Train",i_tr),("Val",i_va),("Test",i_te)]}
r2 = {k: r2_score(y.iloc[ii], np.clip(model_anch.predict(df[ANCHORED].iloc[ii]),0,None)) for k, ii in [("Train",i_tr),("Val",i_va),("Test",i_te)]}
fig, ax = plt.subplots(2, 2, figsize=(10, 7)); xs = list(rmse); xi = np.arange(len(xs)); w = 0.35
ax[0,0].bar(xi-w/2, [rmse[k] for k in xs], w, label="RMSE", color=BLUE); ax[0,0].bar(xi+w/2, [r2[k] for k in xs], w, label="R2", color=ORANGE)
ax[0,0].set_xticks(xi); ax[0,0].set_xticklabels(xs); ax[0,0].legend(fontsize=7); ax[0,0].set_title("Performance by split")
imp = pd.Series(model_anch.feature_importances_, index=ANCHORED).sort_values(); ax[0,1].barh(imp.index, imp.values, color=BLUE); ax[0,1].set_title("XGBoost feature importance (gain)")
pte = np.clip(model_anch.predict(Xte), 0, None)
ax[1,0].scatter(yte, pte, s=8, alpha=0.4, color=BLUE); lim=[0, max(yte.max(), pte.max())]; ax[1,0].plot(lim, lim, "r--", lw=1); ax[1,0].set_xlabel("Actual"); ax[1,0].set_ylabel("Predicted"); ax[1,0].set_title("Actual vs Predicted (test)")
ax[1,1].scatter(pte, yte.values-pte, s=8, alpha=0.4, color=BLUE); ax[1,1].axhline(0, color="red", ls="--", lw=1); ax[1,1].set_xlabel("Predicted"); ax[1,1].set_ylabel("Residual"); ax[1,1].set_title("Residuals (test)")
fig.suptitle("Figure 2 - XGBoost performance & feature importance", fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.97]); _save(fig, "fig2_xgb_performance.png")

# Figure 3 - multi-model CV (reuse MM_NUM)
names = [m["label"] for m in MM_NUM]; r2m = [m["r2m"] for m in MM_NUM]; r2s = [m["r2s"] for m in MM_NUM]; rmm = [m["rmm"] for m in MM_NUM]; rms = [m["rms"] for m in MM_NUM]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(names, r2m, yerr=r2s, capsize=5, color=BLUE, error_kw=dict(ecolor="#333", lw=1.2)); ax[0].set_ylim(0.70, 0.95); ax[0].set_title(f"{K['folds']}-fold CV test R2 (err +/-1 SD)")
ax[1].bar(names, rmm, yerr=rms, capsize=5, color=ORANGE, error_kw=dict(ecolor="#333", lw=1.2)); ax[1].set_title(f"{K['folds']}-fold CV test RMSE (err +/-1 SD)")
for a_, vals, sds in [(ax[0], r2m, r2s), (ax[1], rmm, rms)]:
    for i, (v, s) in enumerate(zip(vals, sds)): a_.text(i, v+s, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Figure 3 - Multi-model comparison (5-fold CV): differences within fold-to-fold noise", fontweight="bold", fontsize=8.8)
fig.tight_layout(rect=[0,0,1,0.95]); _save(fig, "fig3_multimodel.png")

# Figure 4 - ablation (reuse ABL) + Poisson deviance (reuse DEV)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
b1 = ax[0].bar(["With VBT\n(submitted)", "Without VBT\n(ablation)"], [ABL[0], ABL[1]], color=[BLUE, ORANGE]); ax[0].set_ylim(0.70, 0.82); ax[0].set_title("VBT-feature ablation - test R2 (advantage survives)")
b2 = ax[1].bar(["Squared-error\n(submitted)", "Poisson +\nexposure off.", "Poisson +\nVBT off."], list(DEV), color=[ORANGE, BLUE, BLUE]); ax[1].set_title("Poisson deviance (lower is better)")
for a_, bars, vals in [(ax[0], b1, [ABL[0], ABL[1]]), (ax[1], b2, list(DEV))]:
    for r, v in zip(bars, vals): a_.text(r.get_x()+r.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Figure 4 - VBT ablation (left) and principled count formulation (right)", fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.95]); _save(fig, "fig4_ablation_offset.png")

# Figure 5 - SHAP (reuse model_anch)
try:
    import shap
    sub = df[ANCHORED].iloc[i_te].iloc[:K["shap_n"]]
    sv = shap.TreeExplainer(model_anch).shap_values(sub)
    plt.figure(figsize=(8, 5)); shap.summary_plot(sv, sub, show=False, plot_size=None)
    plt.title("Figure 5 - SHAP summary (XGBoost): marginal feature impact", fontweight="bold", fontsize=10)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig5_shap.png"), dpi=170, bbox_inches="tight"); plt.close(); print("fig5_shap.png")
except Exception as e:
    print("SHAP figure skipped:", e)

# Figure 6 - copula aggregate loss (reuse COP)
agg = COP["agg"]
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.hist(agg/1e9, bins=50, color=BLUE, alpha=0.8)
ax.axvline(COP["mean"]/1e9, color="green", ls="--", label=f"mean ${COP['mean']/1e9:.2f}B")
ax.axvline(COP["var99"]/1e9, color="orange", ls="--", label=f"VaR99 ${COP['var99']/1e9:.2f}B")
ax.axvline(COP["tvar99"]/1e9, color="red", ls="--", label=f"TVaR99 ${COP['tvar99']/1e9:.2f}B")
ax.set_xlabel("Aggregate death claims ($B)"); ax.set_ylabel("Simulations"); ax.legend(fontsize=8)
ax.set_title(f"Figure 6 - Copula aggregate-loss distribution (freq-sev rho={COP['rho']:.2f})", fontweight="bold")
fig.tight_layout(); _save(fig, "fig6_copula_aggregate.png")

# Figure 7 - out-of-time (reuse OOT_AE)
labels = ["VBT2015\n(static)", "XGBoost\nanchored", "XGBoost\nno-VBT", "Poisson\nVBT-offset"]
aes = [OOT_AE["vbt"], OOT_AE["anch"], OOT_AE["no"], OOT_AE["off"]]
fig, ax = plt.subplots(figsize=(8, 4.5))
bb = ax.bar(labels, aes, color=[ORANGE if l.startswith("VBT") else BLUE for l in labels])
ax.axhline(1.0, color="red", ls="--", lw=1, label="A/E = 1.0 (perfect)"); ax.set_ylabel("Out-of-time A/E (2018-2019)"); ax.set_ylim(0.8, 1.1); ax.legend(fontsize=8)
for r, v in zip(bb, aes): ax.text(r.get_x()+r.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_title("Figure 7 - Out-of-time calibration: static VBT2015 closest to 1.0", fontweight="bold", fontsize=9)
fig.tight_layout(); _save(fig, "fig7_out_of_time.png")
print("all figures written to", FIG_DIR)

### View the seven figures

Displays the figures inline (they are also saved to `figures/`).

In [ ]:
from IPython.display import Image, display
for _f in sorted(os.listdir(FIG_DIR)):
    if _f.endswith(".png"):
        print(_f); display(Image(filename=os.path.join(FIG_DIR, _f)))

## 10. Where the data comes from

The inputs are derived from the **SOA Individual Life Experience Committee (ILEC)
2012-2019** dataset, published by the Society of Actuaries. The full table holds
**45,501,036 aggregated exposure cells** -- these are exposure cells, not
individual policies, a distinction that matters for interpreting every result
here and one the review committee rightly pressed on.

`data/soa_sample.parquet` is a 131,922-cell extract; `data/full_table_ae.json`
carries actual/expected aggregates computed over the entire table, which is why
Figure 1 can describe the full population while the models train on the sample.
Neither file contains personally identifiable information.

The aggregation step that produced these two files -- the SQL, the column
definitions, the age banding, and the limits on what can be reproduced from the
public extract alone -- is documented in **`docs/DATA_PROVENANCE.md`**. It ran
against a private data warehouse, so it is documented rather than executable
here; every step downstream of it, which is everything in this notebook, runs from
the public files.